# 16-phase6-positional-alpha

**neuron Phase 6** — Phase 5 결론 (per-channel ≈ scalar, channel axis 표현력 한계) 후 paradigm 의
다음 axis 진입. α 를 **위치의 함수** 로 정의:

```
alpha(t)[c] = a_c * sin(t * omega_c) + b_c
```

- **Phase 5 결과** (PR #50): per-channel 의 표현력 우위 미입증, 단 channel-wise implicit pruning 발견 (ch-dead 7.7%, 최대 48%). 결론: channel axis 표현력 확장 한계 → position axis 로 전환.
- **본 Phase 의 가설**:
  1. **position-dependent α 의 task 의미** — TinyShakespeare char-LM 에서 position 0~127 의 attn 기여도 의미 있게 다른가?
  2. **final_loss 우위** — scalar/per_channel 보다 낮은가?
  3. **frequency 학습 활성** — log_freq 가 init 에서 의미 있게 변하는가?
  4. **partial-dead 의 position 의존** — (channel × position) 매트릭스에서 dead 패턴 분포?
- **설계**: 2 × 3 sweep (per_channel baseline + positional × seed {42, 123, 456}) = 6 run, max_steps=1500 (빠른 1차 검증).
- **데이터**: TinyShakespeare (char-LM)
- **시드**: [42, 123, 456]
- **작성일**: 2026-05-25
- **연관**: Issue [#51](https://github.com/EinSofINTEREST/GraphLM/issues/51) / Phase 5 baseline PR [#50](https://github.com/EinSofINTEREST/GraphLM/pull/50)

## 1. 환경 / device

In [ ]:
import logging
import sys

import torch
import torch.nn.functional as F

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.neuron import (
    NeuronConfig,
    NeuronGrowingDecoder,
    SinusoidalAlpha,
    add_attn_smooth_start,
)
from graphlm.training.triggers import PlateauTrigger
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")

## 2. 실험 설정 — per_channel vs positional sweep

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]
ALPHA_MODES = ["per_channel", "positional"]
ALPHA_INIT = 0.10  # Phase 2~5 sweet spot (positional 의 경우 init_bias)

BATCH_SIZE = 16
BLOCK_SIZE = 128
BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_layers=4, n_init_attn=1)
TRAIN = dict(
    lr=3e-4,
    max_steps=1500,
    max_total_attn=8,
    trigger_window=100,
    trigger_epsilon=0.04,
    trigger_cooldown=150,
    trigger_min_history=100,
)
DEAD_THRESHOLD = 0.05
PARTIAL_DEAD_THRESH = 0.80

# 비교 baseline
PHASE4_SCALAR_MEAN = 1.7763
PHASE4_PERCH_MEAN = 1.7778
GRAPHLM_PHASE1_4L_FINAL_LOSS = 1.7871

print(f"SEEDS        = {SEEDS}")
print(f"ALPHA_MODES  = {ALPHA_MODES}")
print(f"ALPHA_INIT   = {ALPHA_INIT} (positional 의 경우 init_bias, init_amp=0)")
print(f"max_steps    = {TRAIN['max_steps']}")
print(f"전체 run  = {len(SEEDS) * len(ALPHA_MODES)} (GPU 약 9분 예상)")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. 2 x 3 sweep 학습

per_channel 모드는 Phase 4/5 와 동일 (재현성 확인용). positional 모드는 SinusoidalAlpha 적용.

In [ ]:
runs = {}
for alpha_mode in ALPHA_MODES:
    per_channel = alpha_mode == "per_channel"
    positional = alpha_mode == "positional"
    cfg = NeuronConfig(
        vocab_size=tokenizer.vocab_size,
        max_seq_len=BLOCK_SIZE,
        alpha_per_channel=per_channel,
        alpha_positional=positional,
        **BACKBONE,
    )
    for seed in SEEDS:
        print(f"--- alpha_mode={alpha_mode}, seed={seed} ---")
        set_seed(seed)
        model = NeuronGrowingDecoder(cfg).to(DEVICE)
        data_iter = iter_random_batches(
            dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=seed
        )
        trigger = PlateauTrigger(
            window=TRAIN["trigger_window"],
            epsilon=TRAIN["trigger_epsilon"],
            cooldown=TRAIN["trigger_cooldown"],
            min_history=TRAIN["trigger_min_history"],
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN["lr"])

        losses = []
        grow_events = []
        next_block_to_grow = 0
        V = cfg.vocab_size
        model.train()
        for step in range(1, TRAIN["max_steps"] + 1):
            x, y = next(data_iter)
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

            if (
                trigger.update(loss.item())
                and sum(model.n_attn_per_block) < TRAIN["max_total_attn"]
            ):
                target_block = next_block_to_grow
                add_attn_smooth_start(model, target_block, alpha_init=ALPHA_INIT)
                block = model.blocks[target_block]
                new_alpha = block.attn_alphas[-1]
                if isinstance(new_alpha, SinusoidalAlpha):
                    new_alpha_params = list(new_alpha.parameters())
                else:
                    new_alpha_params = [new_alpha]
                new_params = (
                    list(block.attn_lns[-1].parameters())
                    + list(block.attns[-1].parameters())
                    + new_alpha_params
                )
                optimizer.add_param_group({"params": new_params, "lr": TRAIN["lr"]})
                grow_events.append((step, target_block, sum(model.n_attn_per_block)))
                next_block_to_grow = (next_block_to_grow + 1) % cfg.n_layers

        if positional:
            final_alphas = []
            for bi, block in enumerate(model.blocks):
                for ai, alpha_mod in enumerate(block.attn_alphas):
                    final_alphas.append(
                        (
                            bi,
                            ai,
                            {
                                "amplitude": alpha_mod.amplitude.detach().cpu().clone(),
                                "bias": alpha_mod.bias.detach().cpu().clone(),
                                "log_freq": alpha_mod.log_freq.detach().cpu().clone(),
                            },
                        )
                    )
        else:
            final_alphas = [
                (bi, ai, alpha_param.detach().cpu().clone())
                for bi, block in enumerate(model.blocks)
                for ai, alpha_param in enumerate(block.attn_alphas)
            ]
        runs[(alpha_mode, seed)] = {
            "losses": losses,
            "grow_events": grow_events,
            "final_alphas": final_alphas,
        }
        n_last = min(100, len(losses))
        final_loss = sum(losses[-n_last:]) / n_last if n_last > 0 else 0.0
        print(f"  done: final_loss(last 100 avg)={final_loss:.4f}, n_added={len(grow_events)}")

        del model, optimizer, trigger, data_iter
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 5. 결과 표 — alpha_mode x seed

positional 의 alpha(t) 는 위치 별 값이 다르므로 dead 측정은 **(position x channel) 매트릭스**
기준. attn-dead 는 전체 (T x hidden) 의 PARTIAL_DEAD_THRESH 이상이 dead 일 때.

In [ ]:
import math
import statistics

import numpy as np

from graphlm.neuron.positional_analysis import eval_positional_alpha

T_EVAL = BLOCK_SIZE


def _eval_pos(a_dict: dict, T: int) -> np.ndarray:
    """노트북 wrapper: dict 입력을 src 함수 시그니처로 unpack."""
    return eval_positional_alpha(
        a_dict["amplitude"].numpy(),
        a_dict["bias"].numpy(),
        a_dict["log_freq"].numpy(),
        T,
    )


print(
    f"{'alpha_mode':>11}  {'seed':>5}  {'n_added':>7}  {'attn-dead%':>10}  "
    f"{'ch-dead%':>9}  {'mean|a|':>8}  {'final_loss':>11}"
)
print("-" * 80)
table = {}
for alpha_mode in ALPHA_MODES:
    table[alpha_mode] = {}
    for seed in SEEDS:
        r = runs[(alpha_mode, seed)]
        added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
        n_added = len(added)
        if n_added == 0:
            attn_dead_pct = math.nan
            ch_dead_pct = math.nan
            mean_abs = math.nan
        elif alpha_mode == "per_channel":
            ch_dead_per_attn = []
            attn_dead_count = 0
            all_abs = []
            for _, _, a in added:
                abs_v = a.abs()
                ch_dead = (abs_v < DEAD_THRESHOLD).float().mean().item()
                ch_dead_per_attn.append(ch_dead)
                if ch_dead >= PARTIAL_DEAD_THRESH:
                    attn_dead_count += 1
                all_abs.extend(abs_v.tolist())
            attn_dead_pct = attn_dead_count / n_added
            ch_dead_pct = statistics.mean(ch_dead_per_attn)
            mean_abs = statistics.mean(all_abs)
        else:
            ch_dead_per_attn = []
            attn_dead_count = 0
            all_abs = []
            for _, _, a_dict in added:
                a_matrix = _eval_pos(a_dict, T_EVAL)
                abs_v = np.abs(a_matrix)
                ch_dead = (abs_v.max(axis=0) < DEAD_THRESHOLD).mean()
                ch_dead_per_attn.append(float(ch_dead))
                if ch_dead >= PARTIAL_DEAD_THRESH:
                    attn_dead_count += 1
                all_abs.extend(abs_v.flatten().tolist())
            attn_dead_pct = attn_dead_count / n_added
            ch_dead_pct = statistics.mean(ch_dead_per_attn)
            mean_abs = statistics.mean(all_abs)

        n_last = min(100, len(r["losses"]))
        final_loss = sum(r["losses"][-n_last:]) / n_last if n_last > 0 else 0.0
        table[alpha_mode][seed] = dict(
            n_added=n_added,
            attn_dead_pct=attn_dead_pct,
            ch_dead_pct=ch_dead_pct,
            mean_abs=mean_abs,
            final_loss=final_loss,
        )
        if n_added == 0:
            attn_str, ch_str, mean_str = "    n/a ", "    n/a ", "    n/a "
        else:
            attn_str = f"{attn_dead_pct:>10.1%}"
            ch_str = f"{ch_dead_pct:>9.1%}"
            mean_str = f"{mean_abs:>8.4f}"
        print(
            f"{alpha_mode:>11}  {seed:>5}  {n_added:>7d}  {attn_str}  {ch_str}  "
            f"{mean_str}  {final_loss:>11.4f}"
        )

## 6. per_channel vs positional 통계 판정 + Phase 4 baseline 비교

In [ ]:
agg = {}
for alpha_mode in ALPHA_MODES:
    fls = [table[alpha_mode][s]["final_loss"] for s in SEEDS]
    agg[alpha_mode] = dict(
        mean=statistics.mean(fls),
        sigma=statistics.stdev(fls) if len(fls) > 1 else 0,
    )

print("=== Phase 6 (1500step): per_channel vs positional ===")
print(
    f"  per_channel mean +/- sigma : {agg['per_channel']['mean']:.4f} +/- {agg['per_channel']['sigma']:.4f}"
)
print(
    f"  positional  mean +/- sigma : {agg['positional']['mean']:.4f} +/- {agg['positional']['sigma']:.4f}"
)
diff = agg["per_channel"]["mean"] - agg["positional"]["mean"]
larger_sigma = max(agg["per_channel"]["sigma"], agg["positional"]["sigma"])
ratio = abs(diff) / larger_sigma if larger_sigma > 0 else float("inf")
print(f"  diff (per_ch - pos)       : {diff:+.4f}")
print(f"  max sigma                 : {larger_sigma:.4f}")
print(f"  |diff| / sigma             : {ratio:.2f}")
if ratio < 1.0:
    verdict = "[neutral] 통계적으로 무의미 (|diff| < sigma) — positional 의 추가 표현력 미입증"
elif diff > 0:
    # diff = per_ch.mean - pos.mean > 0  =>  positional 의 loss 가 더 낮음  =>  positional 우위
    verdict = "[ok] positional 명확 우위 (|diff| > sigma) — position dynamics 가 task 에 의미"
else:
    verdict = "[neg] per_channel 우위 (|diff| > sigma) — positional 의 자유도가 noise"
print(f"  verdict                   : {verdict}")
n_pos_better = sum(
    1 for s in SEEDS if table["positional"][s]["final_loss"] < table["per_channel"][s]["final_loss"]
)
print(f"  positional < per_ch       : {n_pos_better}/{len(SEEDS)} 시드")

print()
print("=== Phase 4 baseline 대비 (Phase 6 1500step) ===")
print(f"  Phase 4 scalar (1500step)      : {PHASE4_SCALAR_MEAN:.4f}")
print(f"  Phase 4 per_channel (1500step) : {PHASE4_PERCH_MEAN:.4f}")
print(f"  Phase 6 per_channel (재현)     : {agg['per_channel']['mean']:.4f}")
print(f"  Phase 6 positional             : {agg['positional']['mean']:.4f}")

## 7. positional alpha 의 amplitude/bias/log_freq 분석

핵심 질문: **frequency 가 init 에서 의미 있게 변했는가?** 만약 거의 안 변했으면 sinusoidal 의
가치 미입증 (그냥 bias 만 작동).

In [ ]:
print("=== positional alpha 학습 진화 (init 대비 amplitude / log_freq 변화) ===")
init_log_freqs = np.linspace(math.log(0.01), math.log(1.0), BACKBONE["hidden_dim"])

print(
    f"{'seed':>5}  {'added':>5}  {'block':>5}  "
    f"{'|amp| mean':>10}  {'|amp| max':>9}  "
    f"{'log_freq d mean':>15}  {'|bias| mean':>11}  {'ch-dead%':>8}"
)
print("-" * 95)
for seed in SEEDS:
    r = runs[("positional", seed)]
    added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
    for idx, (bi, _ai, a_dict) in enumerate(added):
        amp = a_dict["amplitude"].numpy()
        bias = a_dict["bias"].numpy()
        log_freq = a_dict["log_freq"].numpy()
        a_matrix = _eval_pos(a_dict, T_EVAL)
        ch_dead = (np.abs(a_matrix).max(axis=0) < DEAD_THRESHOLD).mean()
        log_freq_delta = np.abs(log_freq - init_log_freqs).mean()
        print(
            f"{seed:>5}  {idx:>5}  {bi:>5}  "
            f"{np.abs(amp).mean():>10.4f}  {np.abs(amp).max():>9.4f}  "
            f"{log_freq_delta:>15.4f}  {np.abs(bias).mean():>11.4f}  {ch_dead:>8.1%}"
        )

## 8. 시각화 — alpha(t, c) heatmap + frequency 분포 + final_loss 비교

(a) seed 42 positional 의 added attn 1개 alpha(t, c) heatmap.
(b) added attn 들의 log_freq init vs final 분포.
(c) per_channel vs positional final_loss bar.

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1.3, 1])

# (a) alpha(t, c) heatmap
ax_a = fig.add_subplot(gs[0, 0])
r = runs[("positional", SEEDS[0])]
added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
if added:
    bi0, _ai0, a_dict0 = added[0]
    a_mat = _eval_pos(a_dict0, T_EVAL)
    vmax = max(np.abs(a_mat).max(), 0.1)
    im = ax_a.imshow(a_mat.T, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax_a.set_xlabel("position t")
    ax_a.set_ylabel("channel c")
    ax_a.set_title(f"alpha(t, c) — seed={SEEDS[0]}, block={bi0}, added#0")
    fig.colorbar(im, ax=ax_a, fraction=0.046, pad=0.04)

# (b) log_freq init vs final
ax_b = fig.add_subplot(gs[0, 1])
ax_b.hist(init_log_freqs, bins=30, alpha=0.5, label="init (log-spaced)", color="gray")
for idx, (bi, _ai, a_dict) in enumerate(added):
    ax_b.hist(
        a_dict["log_freq"].numpy(),
        bins=30,
        alpha=0.5,
        label=f"final added#{idx} block={bi}",
    )
ax_b.set_xlabel("log_freq")
ax_b.set_ylabel("count of channels")
ax_b.set_title(f"log_freq 분포: init vs final (seed={SEEDS[0]})")
ax_b.legend(loc="upper left", fontsize=8)
ax_b.grid(alpha=0.3, axis="y")

# (c) final_loss bar
ax_c = fig.add_subplot(gs[1, :])
width = 0.35
x_seeds = list(range(len(SEEDS)))
all_final = [table[m][s]["final_loss"] for m in ALPHA_MODES for s in SEEDS]
colors = {"per_channel": "#ff7f0e", "positional": "#2ca02c"}
for i, alpha_mode in enumerate(ALPHA_MODES):
    offsets = [x + (i - 0.5) * width for x in x_seeds]
    vals = [table[alpha_mode][s]["final_loss"] for s in SEEDS]
    ax_c.bar(offsets, vals, width=width, color=colors[alpha_mode], label=alpha_mode, alpha=0.85)
ax_c.axhline(
    PHASE4_SCALAR_MEAN,
    color="#1f77b4",
    linestyle=":",
    lw=1,
    label=f"Phase 4 scalar mean ({PHASE4_SCALAR_MEAN})",
)
ax_c.axhline(
    PHASE4_PERCH_MEAN,
    color="#ff7f0e",
    linestyle=":",
    lw=1,
    label=f"Phase 4 per_channel mean ({PHASE4_PERCH_MEAN})",
)
ax_c.axhline(
    GRAPHLM_PHASE1_4L_FINAL_LOSS,
    color="red",
    linestyle="--",
    lw=1,
    label=f"GraphLM 4L baseline ({GRAPHLM_PHASE1_4L_FINAL_LOSS})",
)
ax_c.set_xlabel("seed")
ax_c.set_xticks(x_seeds)
ax_c.set_xticklabels([str(s) for s in SEEDS])
ax_c.set_ylabel("final loss (last 100 avg)")
ax_c.set_title("Phase 6 final loss: per_channel vs positional + Phase 4 baselines")
ax_c.legend(loc="upper right", fontsize=8)
ax_c.grid(alpha=0.3, axis="y")
ymin_d = min(min(all_final), PHASE4_SCALAR_MEAN, PHASE4_PERCH_MEAN)
ymax_d = max(max(all_final), GRAPHLM_PHASE1_4L_FINAL_LOSS)
margin = max(0.02, (ymax_d - ymin_d) * 0.15)
ax_c.set_ylim(ymin_d - margin, ymax_d + margin)

fig.tight_layout()
fig.savefig(OUT_DIR / "positional_alpha.png", dpi=120)
plt.show()

## 결과 요약 / Phase 7 권장 방향

확인 포인트:
- §6 verdict — positional 이 per_channel 보다 명확히 낮은가? (|diff|/sigma >= 1.0)
- §7 log_freq d — frequency 가 실제로 학습되었는가? (~0 이면 sinusoidal 가치 미입증)
- §8 (a) heatmap — position 별 alpha 패턴이 random 인가 structured 인가?
- §8 (b) log_freq histogram — init log-spaced 분포가 학습 후 변형되었는가?

**판정 시나리오**:
- **A. positional 명확 우위 + frequency 학습 활성** — position dynamics 가 task 에 의미. Phase 7: MLP-based alpha 비교 / 더 큰 task / max_steps 5000 재검증
- **B. positional ≈ per_channel (|diff|<sigma)** — gating 함수의 표현력 한계. Phase 7: Phase 6+ 후보 다음 (nn.Parameter 정적 shape 초월, in-place expansion)
- **C. positional 열위 + frequency 안 학습** — sinusoidal 자체 한계. MLP-based 또는 T5-style relative bias 로 재시도
- **D. positional 열위 + frequency 학습 활성** — position dynamics 가 noise. Phase 5 의 implicit pruning 처럼 다른 메커니즘 탐색

**참고**:
- Phase 5 결과 (Notion): https://www.notion.so/36be8b70b7aa812cae39d41a3660f0e7
- Phase 6+ 후보 (다음 주제): https://www.notion.so/36be8b70b7aa8142bebdc9706c64ed52